# Bring your own CSV or Parquet file

## Goal

Save a table with its column meanings, select useful rows, and keep the same
workflow for CSV and Parquet. Our question is: **Which fictional areas recorded
at least five visits in 2024?** Allow about 15 minutes.

Every code, year and count below is independently invented teaching material.
Nothing is downloaded. A four-row table lets us check every result directly.

## Setup

Read the results here, or follow [installation](https://xangma.github.io/ssdu/start/install.html)
and open this notebook in JupyterLab. Run code cells in order with **Shift+Enter**.
This notebook needs only the installed SSDU package and its normal dependencies;
there are no companion files to find. It creates files inside `data/local-files-lesson`.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

from ssdu import DatasetQuerySpec, LocalImportSpec, Service, Store
from ssdu.models import RetrievalError

workspace = Path("data/local-files-lesson")
workspace.mkdir(parents=True, exist_ok=True)
store = Store(workspace / "saved-data")

## Steps

### 1. Inspect a small CSV

CSV is a plain-text table. The area codes below are **text identifiers**: `001`
must keep its leading zeros. A blank count means no count was reported; `0` is a
reported zero. These have different meanings.

In [ ]:
source_file = workspace / "visits.csv"
source_file.write_text(
    "area_code,year,visits\n001,2024,12\n002,2024,\n003,2024,0\n004,2024,7\n",
    encoding="utf-8",
)
print(source_file.read_text(encoding="utf-8"))

### 2. Describe what one row means

One row represents one fictional area in one year. Together, `area_code` and
`year` identify a row: they form its **primary key**, which must be unique and
non-missing. Declare codes as `string` (text) and counts as `int64` (whole numbers).
CSV columns otherwise default to text.

`null_values` turns the exact blank CSV token into a missing value.
`missing_values` records the explanation; it does not perform that conversion.

In [ ]:
details = LocalImportSpec(
    dataset_id="invented-service-visits",
    title="Invented service visits, 2024",
    description="Four independently invented rows for a local-file lesson.",
    citation="SSDU CSV/Parquet lesson, invented fixture version 1; not official statistics.",
    licence="MIT — independently invented SSDU teaching material",
    row_grain="One row per fictional area and year.",
    primary_key=["area_code", "year"],
    null_values=[""],
    columns={
        "area_code": {"type": "string", "definition": "Fictional area identifier"},
        "year": {"type": "int64", "definition": "Year described by the count"},
        "visits": {
            "type": "int64",
            "definition": "Number of visits to the fictional service",
            "unit": "visits",
            "missing_values": {"": "No count reported; do not replace with zero"},
        },
    },
)
csv_dataset = store.import_dataset(source_file, details)
print(csv_dataset.to_pandas().to_string(index=False, na_rep="missing"))

You should see all four rows, with codes `001`–`004` unchanged. Area `002` has a
missing count, while `003` has zero. SSDU saved the original file, the typed table
and its descriptions together in a **snapshot**. Editing the original CSV later
does not change that snapshot.

### 3. Inspect the description and select rows

Check the saved meaning before using the values. A `records` profile stores this
typed table; provider datasets can use `records` or `observations`, depending on
what they contain.

In [ ]:
print("Profile:", csv_dataset.contract["profile"])
print("One row:", csv_dataset.contract["row_grain"])
print("Citation:", csv_dataset.contract["source"]["citation"])
print("Saved types:", csv_dataset.to_arrow().schema)

The saved types should include text codes and whole-number years and visits.
Now select rows for 2024 with at least five visits. `eq` means equals and `gte`
means greater than or equal to. Missing counts do not pass this numeric filter.

In [ ]:
service = Service(store.root)
selection = DatasetQuerySpec(
    columns=["area_code", "year", "visits"],
    filters=[
        {"column": "year", "op": "eq", "value": 2024},
        {"column": "visits", "op": "gte", "value": 5},
    ],
    sort=[{"column": "area_code"}],
    limit=10,
)
result = service.query_dataset(csv_dataset.snapshot_id, selection)
print(pd.DataFrame(result["rows"]).to_string(index=False))

Only **001 (12 visits)** and **004 (7 visits)** should appear. The zero and the
missing count are both absent, for different reasons. These are counts, not
rates: comparing service use would also need a population or other denominator.

### 4. Use the same workflow for Parquet

Parquet is a file format that stores column types and missing values. We create
an invented Parquet version from the saved table, so no binary fixture is needed.
For an existing file, point `parquet_file` at your own `.parquet` path instead of
running the line that writes it.

In [ ]:
parquet_file = workspace / "visits.parquet"
pq.write_table(csv_dataset.to_arrow(), parquet_file)
parquet_details = details.model_copy(update={"dataset_id": "invented-service-visits-parquet"})
parquet_dataset = store.import_dataset(parquet_file, parquet_details)
parquet_result = service.query_dataset(parquet_dataset.snapshot_id, selection)
print(pd.DataFrame(parquet_result["rows"]).to_string(index=False))

The same two rows should appear. Parquet keeps native types and nulls; CSV's
`null_values` setting does not reinterpret Parquet text. You still supply column
meanings, units, keys, citation and rights. Each format has its own snapshot ID;
equal rows do not mean the original file bytes are identical.

### 5. Recognise a useful validation error

A whole-number count should not silently accept the word `many`. This cell writes
a separate invalid teaching file and shows SSDU rejecting it. Your valid snapshot
stays available.

In [ ]:
invalid_file = workspace / "invalid-visits.csv"
invalid_file.write_text("area_code,year,visits\n001,2024,many\n", encoding="utf-8")
try:
    store.import_dataset(invalid_file, details)
except RetrievalError as error:
    validation_message = str(error)
    print(validation_message)
else:
    raise AssertionError("The invalid count should have been rejected")

Expect `Column 'visits' contains invalid int64 values`. Check the source value and
its meaning before correcting it. Removing the type declaration would keep text,
but would not make `many` a usable count. If keys are duplicated, check whether
the row grain needs another identifier, such as service type, before importing.

## Checks

Confirm the selected rows, the preserved leading zeros and null, and the matching
CSV/Parquet tables. We also reopen the CSV snapshot from its saved copy.

In [ ]:
expected_rows = [
    {"area_code": "001", "year": 2024, "visits": 12},
    {"area_code": "004", "year": 2024, "visits": 7},
]
assert result["rows"] == parquet_result["rows"] == expected_rows
assert csv_dataset.to_arrow().column("area_code").to_pylist() == ["001", "002", "003", "004"]
assert csv_dataset.to_arrow().column("visits").to_pylist() == [12, None, 0, 7]
assert csv_dataset.to_arrow().equals(parquet_dataset.to_arrow())
assert validation_message == "Column 'visits' contains invalid int64 values"
reopened = Store(workspace / "saved-data").open_dataset(csv_dataset.snapshot_id)
assert reopened.to_arrow().equals(csv_dataset.to_arrow())
print("Checks passed: types, missing values, filters and saved copies agree.")

## Next Steps

To adapt this recipe, **skip the cells that create invented input files**. Set
`source_file = Path("/path/to/your.csv")` or
`parquet_file = Path("/path/to/your.parquet")`, then update `details` for your file:

- Match the real column names and types; keep identifiers as text.
- Describe one row and choose keys that are unique and non-missing.
- Set the unit and missing-value meanings, plus the original source, release and
  licence. Use the real rights information rather than copying this fixture's MIT licence.
- Import, inspect the table and change the query's filters to your question.

If a CSV uses a missing marker such as `..`, add it to `null_values` only after
checking its meaning. That setting applies to every CSV column. Keep any
suppression explanation in `missing_values`. Imports are bounded to 25 MB of
file bytes and 100,000 rows by default; split or prepare a larger input deliberately.

Continue with [dataset queries](https://xangma.github.io/ssdu/guides/dataset-api.html)
or [transfer saved data to another machine](https://xangma.github.io/ssdu/start/quickstart.html).